In [ ]:
#pip install lime

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import set_random_seed

import plotnine as p9
import shap
from lime.lime_tabular import LimeTabularExplainer

pd.options.display.float_format = '{:.6f}'.format

In [ ]:
set_random_seed(1234)
n = 800  # increase to 1000 if needed

# Generate data
v1 = np.random.uniform(-1, 1, n)
v2 = np.random.uniform(-1, 1, n)
v3 = np.random.uniform(-1, 1, n)

# Define response variable y
y = np.where(v1 < 0, 0,
             np.where(abs(v2) > 1/4, 1, 0))
# y = y.astype("category")

# Create dataframe
dfc = pd.DataFrame({"v1": v1, "v2": v2, "v3": v3, "y": y})

# Create factor-like variables (boolean → categorical)
z1 = (dfc["v1"] > 0).astype("category")
z2 = (abs(dfc["v2"]) > 1/4).astype("category")

dfc = pd.DataFrame({"v1": v1, "v2": v2, "v3": v3, "y": y, "z1": z1, "z2": z2})

# --- dataset ---
X = dfc[["v1", "v2", "v3"]].values  # as NumPy array
y = dfc["y"]#.cat.codes              # convert categorical to numeric codes (0,1,...)

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v1, y = v2, color = y.astype("category"))) +
    p9.geom_point()
)

gg

In [ ]:
# --- Define MLP model builder ---
def build_mlp(input_dim):
    model = Sequential([
        Dense(64, activation="relu", input_shape=(input_dim,)),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=Adam(learning_rate=0.01),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

In [ ]:
# --- Define MLP model builder ---
def build_mlp(input_dim):
    model = Sequential([
        Dense(128, activation="relu", input_shape=(input_dim,)),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=Adam(learning_rate=0.01),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

In [ ]:
def clique(X, y, model_builder="rf", folds=5, nsim=26, quantile_grid=True, keras_model=False, 
            epochs=10, batch_size=32, random_state=123):
    """
    Python translation of the R clique function.
    
    Parameters
    ----------
    X : pd.DataFrame
        Predictor matrix
    y : pd.Series or np.array
        Response vector (numeric for regression, categorical for classification)
    model_type : str
        "rf" for random forest, "lm" for linear regression
    param_grid : dict
        Parameters for the chosen model (like tuneGrid in caret)
    nsim : int
        Number of grid points
    folds : int
        Number of CV folds
    quantile_grid : bool
        Whether to use quantiles (True) or uniform grid (False)
    random_state : int
        Reproducibility
    
    Returns
    -------
    dict with:
        - "models": list of fitted models (per fold)
        - "local_imp": pd.DataFrame of local importance values
    """
        
    X = pd.DataFrame(X)
    y = pd.Series(y)

    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=random_state)
    models = []
    
    # Train fold-specific models
    for train_idx, test_idx in skf.split(X, y):
        if model_builder == "rf":
            model = RandomForestClassifier(n_estimators=100, random_state=random_state)
        elif keras_model:
            model = model_builder(X.shape[1])  # build keras model
        else:
            model = model_builder  # assume it's already an sklearn estimator

        if keras_model:
            model.fit(X.iloc[train_idx], y.iloc[train_idx],
                      epochs=epochs, batch_size=batch_size, verbose=0,
                      validation_data=(X.iloc[test_idx], y.iloc[test_idx]))
        else:
            model.fit(X.iloc[train_idx], y.iloc[train_idx])

        models.append(model)

    # Collect predictions (truth vs. preds)
    truth = y.values
    preds = np.zeros_like(truth, dtype=float)

    for fold, (_, test_idx) in enumerate(skf.split(X, y)):
        if keras_model:
            preds[test_idx] = models[fold].predict(X.iloc[test_idx]).ravel()
            preds[test_idx] = (preds[test_idx] > 0.5).astype(int)  # convert to binary
        else:
            preds[test_idx] = models[fold].predict_proba(X.iloc[test_idx])[:, 1]
            preds[test_idx] = (preds[test_idx] > 0.5).astype(int)  # convert to binary

    base_mse = abs(truth - preds)

    # Importance calculation
    local_imp = pd.DataFrame(0, index=X.index, columns=X.columns)

    for i, col in enumerate(X.columns):
        if np.issubdtype(X[col].dtype, np.number):
            grid_vals = np.quantile(X[col], np.linspace(0, 1, nsim)) if quantile_grid else np.linspace(X[col].min(), X[col].max(), nsim)
        else:
            grid_vals = X[col].unique()

        for val in grid_vals:
            X_new = X.copy()
            X_new[col] = val
            new_preds = np.zeros_like(truth, dtype=float)

            for fold, (_, test_idx) in enumerate(skf.split(X, y)):
                if keras_model:
                    new_preds[test_idx] = models[fold].predict(X_new.iloc[test_idx]).ravel()
                    new_preds[test_idx] = (new_preds[test_idx] > 0.5).astype(int)  # convert to binary
                else:
                    new_preds[test_idx] = models[fold].predict_proba(X_new.iloc[test_idx])[:, 1]
                    new_preds[test_idx] = (new_preds[test_idx] > 0.5).astype(int)  # convert to binary

            new_mse = abs(truth - new_preds)
            local_imp[col] += (new_mse - base_mse) / nsim

    return {"models": models, "local_imp": local_imp}

In [ ]:
from sklearn.metrics import log_loss  # or other loss

def ici(model, X, y, feature_idx, nsim = 25, scoring=log_loss):
    """Compute a local importance for one observation and one feature."""
    baseline_pred = model.predict(X).ravel()  # shape (1, n_classes)
    baseline_loss = (y - baseline_pred) ** 2
    
    new_loss = np.zeros((nsim, len(baseline_loss)))
    for i in range(nsim):
        X_permuted = X.copy()
        y_permuted = y.copy()

        other_idx = [i for i in range(X.shape[1]) if i != feature_idx]
        perm = np.random.permutation(X.shape[0])
        X_permuted[:, other_idx] = X[perm][:, other_idx]
        y_permuted = y[perm]
        #X_permuted[:, feature_idx] = np.random.choice(X[:, feature_idx], size=1)
        new_pred = model.predict(X_permuted).ravel()
        new_loss[i, :] = (y_permuted - new_pred) ** 2

    new_loss = np.mean(new_loss, axis=0)
    return new_loss - baseline_loss


In [ ]:
res = clique(X=pd.DataFrame(X), y=pd.Series(y), model_builder="rf", nsim=26, folds=5)

In [ ]:
res["local_imp"].describe().drop(["count", "mean", "std"])

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v1, y = res["local_imp"][0], color = z2)) +
    p9.geom_point()
    )

gg

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v2, y = res["local_imp"][1], color = z1)) +
    p9.geom_point()
    )

gg

In [ ]:
res_mlp = clique(X, y, model_builder=build_mlp, quantile_grid=True,
                  folds=5, nsim=25, keras_model=True, epochs=100, batch_size=64)

In [ ]:
pd.options.display.float_format = '{:.6f}'.format
res_mlp["local_imp"]

In [ ]:
res_mlp["local_imp"].describe().drop(["count", "mean", "std"])

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v1, y = res_mlp["local_imp"][0], color = z2)) +
    p9.geom_point()
    )

gg

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v1, y = res_mlp["local_imp"][0], 
                          color = (abs(dfc["v2"]) > 0.22) & (abs(dfc["v2"]) < 0.28))) +
    p9.geom_point()
    )

gg

In [ ]:
#res_mlp["local_imp"][1]
dfc[res_mlp["local_imp"][0] < 0]

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v2, y = res_mlp["local_imp"][1], color = z1)) +
    p9.geom_point()
    )

gg

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v2, y = res_mlp["local_imp"][1], color = abs(dfc["v1"]) < 0.05)) +
    p9.geom_point()
    )

gg

In [ ]:
#res_mlp["local_imp"][1]
dfc[res_mlp["local_imp"][1] < 0]

In [ ]:
Xdf = pd.DataFrame(X)
ydf = pd.Series(y)

ann_model = build_mlp(Xdf.shape[1])  # build keras model
kf = KFold(n_splits=3, shuffle=True, random_state=123)
for train_idx, test_idx in kf.split(Xdf):
    ann_model.fit(Xdf.iloc[train_idx], ydf.iloc[train_idx], epochs=100, batch_size=64,
              verbose=0, validation_data=(Xdf.iloc[test_idx], ydf.iloc[test_idx]))



In [ ]:
explainer_shap = shap.Explainer(ann_model.predict, Xdf.values)
shap_values = explainer_shap(Xdf.values)

In [ ]:
shap_values.shape

# Extract SHAP values for the first two variables
shap_v1 = shap_values.values[:, 0]
shap_v2 = shap_values.values[:, 1]

In [ ]:
def predict_proba(X):
    # Get probability of positive class (output of sigmoid)
    p1 = ann_model.predict(X)
    # Compute complementary probability for class 0
    p0 = 1 - p1
    # Concatenate into 2D array [p0, p1]
    return np.hstack((p0, p1))


In [ ]:
explainer = LimeTabularExplainer(
    training_data = X,
    class_names = ["Class 0", "Class 1"],
    mode = "classification"
)

feature_names = [f"Feature_{i}" for i in range(3)]

exp = explainer.explain_instance(X[0], predict_proba)
exp.show_in_notebook(show_table=True)

exp

In [ ]:
local_exp = dict(exp.as_list())
local_exp.values()

In [ ]:
explainer = LimeTabularExplainer(
    training_data = X,
    class_names = ["Class 0", "Class 1"],
    mode = "classification"
)

feature_names = [f"Feature_{i}" for i in range(3)]

lime_values = []
for i in range(len(X)):
    exp = explainer.explain_instance(X[i], predict_proba)
    local_exp = dict(exp.as_list())
    row = local_exp.values()
    lime_values.append(row)

lime_df = pd.DataFrame(lime_values, columns=feature_names)

In [ ]:
lime_df

In [ ]:
# Build ici matrix
n_obs, n_feats = X.shape
ici_imp = np.zeros((n_obs, n_feats))
for j in range(n_feats):
    ici_imp[:, j] = ici(ann_model, X, y, j, nsim = 25)

import pandas as pd
ici_df = pd.DataFrame(ici_imp, columns=[f"f{j}" for j in range(n_feats)])

In [ ]:
# Extract LIME values for the first two variables
lime_v1 = lime_df["Feature_0"].values
lime_v2 = lime_df["Feature_1"].values

# --- ICI values ---
ici_v1 = ici_df["f0"].values
ici_v2 = ici_df["f1"].values

In [ ]:
# Create the first dataframe for the first column of importance values and original values
df_v1 = pd.DataFrame({
    "v1": v1,
    "Clique_v1": res_mlp["local_imp"][0],
    "ICI_v1": ici_v1,
    "SHAP_v1": shap_v1,
    "LIME_v1": lime_v1,
    "z2": z2
})

# Create the second dataframe for the second column of importance values and original values
df_v2 = pd.DataFrame({
    "v2": v2,
    "Clique_v2": res_mlp["local_imp"][1],
    "ICI_v2": ici_v2,
    "SHAP_v2": shap_v2,
    "LIME_v2": lime_v2,
    "z1": z1
})



In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v1, y = lime_v1, color = z2)) +
    p9.geom_point()
    )

gg

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v2, y = lime_v2, color = z1)) +
    p9.geom_point()
    )

gg

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v1, y = shap_v1, color = z2)) +
    p9.geom_point()
    )

gg

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v2, y = shap_v2, color = z1)) +
    p9.geom_point()
    )

gg

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v1, y = ici_v1, color = z2)) +
    p9.geom_point()
    )

gg

In [ ]:
gg = (
    p9.ggplot(dfc, p9.aes(x = v2, y = ici_v2, color = z1)) +
    p9.geom_point()
    )

gg

In [ ]:
gg = (
    p9.ggplot(df_v2, p9.aes(x = v2, y = shap_v2, color = z1)) +
    p9.geom_point()
    )

gg

In [ ]:
# Write dfc to a CSV file
dfc.to_csv("dfc_r.csv", index=False)

# Write df_v1 to a CSV file
df_v1.to_csv("df_v1_r.csv", index=False)

# Write df_v2 to a CSV file
df_v2.to_csv("df_v2_r.csv", index=False)

In [ ]:
# --- 2-Fold Cross Validation ---
kf = KFold(n_splits=2, shuffle=True, random_state=42)
fold_accuracies = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"\n--- Fold {fold+1} ---")
    
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # Build and train model
    model = build_mlp(input_dim=X.shape[1])
    model.fit(X_train, y_train,
              validation_data=(X_test, y_test),
              epochs=10,
              batch_size=32,
              verbose=0)
    
    # Evaluate
    y_pred = (model.predict(X_test) > 0.5).astype(int).ravel()
    acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(acc)
    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

# --- Mean accuracy across folds ---
print("\nMean CV Accuracy:", np.mean(fold_accuracies).round(4))

MC Simulations

In [ ]:
n = 800 # increase to 1000 if needed

m = 50

mae_v1 = pd.DataFrame(np.zeros((m, 5)),
                      columns=["v1", "LIME", "SHAP", "ICI", "Clique"])
mae_v2 = pd.DataFrame(np.zeros((m, 5)),
                      columns=["v2", "LIME", "SHAP", "ICI", "Clique"])

for k in range(m):
    set_random_seed(k)

    # Generate data
    v1 = np.random.uniform(-1, 1, n)
    v2 = np.random.uniform(-1, 1, n)
    v3 = np.random.uniform(-1, 1, n)

    # Define response variable y
    y = np.where(v1 < 0, 0,
                np.where(abs(v2) > 1/4, 1, 0))
    # y = y.astype("category")

    # Create dataframe
    dfc = pd.DataFrame({"v1": v1, "v2": v2, "v3": v3, "y": y})

    # Create factor-like variables (boolean → categorical)
    z1 = (dfc["v1"] > 0).astype("category")
    z2 = (abs(dfc["v2"]) > 1/4).astype("category")

    dfc = pd.DataFrame({"v1": v1, "v2": v2, "v3": v3, "y": y, "z1": z1, "z2": z2})

    # --- dataset ---
    X = dfc[["v1", "v2", "v3"]].values  # as NumPy array
    y = dfc["y"]#.cat.codes              # convert categorical to numeric codes (0,1,...)

    res_mlp = clique(X, y, model_builder=build_mlp, quantile_grid=True,
                  folds=5, nsim=25, keras_model=True, epochs=100, batch_size=64)
    
    Xdf = pd.DataFrame(X)
    ydf = pd.Series(y)

    ann_model = build_mlp(Xdf.shape[1])  # build keras model
    kf = KFold(n_splits=3, shuffle=True, random_state=123)
    for train_idx, test_idx in kf.split(Xdf):
        ann_model.fit(Xdf.iloc[train_idx], ydf.iloc[train_idx], epochs=100, batch_size=64,
                verbose=0, validation_data=(Xdf.iloc[test_idx], ydf.iloc[test_idx]))
        
    explainer_shap = shap.Explainer(ann_model.predict, Xdf.values)
    shap_values = explainer_shap(Xdf.values)
    shap_v1 = shap_values.values[:, 0]
    shap_v2 = shap_values.values[:, 1]

    explainer = LimeTabularExplainer(
        training_data = X,
        class_names = ["Class 0", "Class 1"],
        mode = "classification"
    )

    feature_names = [f"Feature_{i}" for i in range(3)]

    lime_values = []
    for i in range(len(X)):
        exp = explainer.explain_instance(X[i], predict_proba)
        local_exp = dict(exp.as_list())
        row = local_exp.values()
        lime_values.append(row)

    lime_df = pd.DataFrame(lime_values, columns=feature_names)

    n_obs, n_feats = X.shape
    ici_imp = np.zeros((n_obs, n_feats))
    for j in range(n_feats):
        ici_imp[:, j] = ici(ann_model, X, y, j, nsim = 25)

    ici_df = pd.DataFrame(ici_imp, columns=[f"f{j}" for j in range(n_feats)])

    # Extract LIME values for the first two variables
    lime_v1 = lime_df["Feature_0"].values
    lime_v2 = lime_df["Feature_1"].values

    # --- ICI values ---
    ici_v1 = ici_df["f0"].values
    ici_v2 = ici_df["f1"].values

    # Create the first dataframe for the first column of importance values and original values
    d1 = pd.DataFrame({
        "v1": v1,
        "LIME": lime_v1,
        "SHAP": shap_v1,
        "ICI": ici_v1,
        "Clique": res_mlp["local_imp"][0],
        "z2": z2
    })

    # 1. sc: max absolute value per column (excluding 6th column)
    sc = d1.drop(d1.columns[5], axis=1).abs().max()

    # 2. vr: filter rows where z2 == "False", then drop z2 column
    vr = d1[d1["z2"] == False].drop(columns=["z2"])

    # 3. mae1: column-wise mean of abs(vr / sc)
    mae_v1.loc[k, :] = (vr.divide(sc, axis=1).abs()).mean().values
    mae_v1.to_csv("mae_v1.csv", index=False)
    
    # Create the second dataframe for the second column of importance values and original values
    d2 = pd.DataFrame({
        "v2": v2,
        "LIME": lime_v2,
        "SHAP": shap_v2,
        "ICI": ici_v2,
        "Clique": res_mlp["local_imp"][1],
        "z1": z1
    })

    # 1. sc: max absolute value per column (excluding 6th column)
    sc = d2.drop(d2.columns[5], axis=1).abs().max()

    # 2. vr: filter rows where z1 == "False", then drop z1 column
    vr = d2[d2["z1"] == False].drop(columns=["z1"])
    
    # 3. mae1: column-wise mean of abs(vr / sc)
    mae_v2.loc[k, :] = (vr.divide(sc, axis=1).abs()).mean().values
    mae_v2.to_csv("mae_v2.csv", index=False)

    # print(k)
    # print(mae_v1.iloc[k, :])
    # print(mae_v2.iloc[k, :])

In [ ]:
mae_v1